# AML Graph Autoencoder - End-to-End Pipeline with Deep Explainability
This notebook demonstrates the unified AML pipeline: Data Simulation -> Heuristic Graph Construction -> GATv2 Training -> SHAP & Network Interpretation.

In [ ]:
import os, sys
sys.path.append(os.path.abspath("../"))
import yaml, torch, pandas as pd, numpy as np
from sklearn.preprocessing import StandardScaler
from src.utils.reproducibility import set_seed, save_checkpoint, load_checkpoint, get_device
from src.data.generator import generate_transactions
from src.graph.builder import GraphBuilder
from src.models.gnn import AMLGraphAutoencoder, compute_combined_loss
from src.explain.interpreter import AnomalyInterpreter

# 0. Setup
with open('../config/config.yaml', 'r') as f:
    config = yaml.safe_load(f)
set_seed(42)
device = get_device(config)

os.makedirs('../artifacts', exist_ok=True)
os.makedirs('../output', exist_ok=True)

# 1. Data Generation & Feature Engineering
NUM_CUSTOMERS = config['data']['num_customers']
full_tx_df = generate_transactions(num_customers=NUM_CUSTOMERS, num_days=40, anomaly_ratio=0.05)
c_id_col = config['data']['column_mapping']['customer_id']
full_tx_df = full_tx_df[full_tx_df[c_id_col] < NUM_CUSTOMERS]

# Add Custom Feature Example
full_tx_df['vol_log'] = np.log1p(full_tx_df['amount'])
config['graph']['node_features'] = ['vol_log']

train_df = full_tx_df[full_tx_df['date'] < '2026-02-01'].copy()
oot_df = full_tx_df[full_tx_df['date'] >= '2026-02-01'].copy()

# 2. Build Training Graph
builder = GraphBuilder(config)
train_data = builder.build_graph(train_df)

# 3. Preprocessing
node_scaler = StandardScaler()
edge_scaler = StandardScaler()
x_scaled = node_scaler.fit_transform(train_data.x.numpy())
edge_attr_scaled = edge_scaler.fit_transform(train_data.edge_attr.numpy())

train_data.x = torch.from_numpy(x_scaled).float().to(device)
train_data.edge_index = train_data.edge_index.to(device)
train_data.edge_attr = torch.from_numpy(edge_attr_scaled).float().to(device)

# 4. Training
model = AMLGraphAutoencoder(train_data.num_node_features, train_data.num_edge_features, config).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

history = {'Total Loss': [], 'Node MSE': [], 'Edge MSE': []}
model.train()
for epoch in range(101):
    optimizer.zero_grad()
    z, x_recon, edge_recon = model(train_data.x, train_data.edge_index, train_data.edge_attr)
    loss, n_mse, e_mse = compute_combined_loss(train_data.x, x_recon, train_data.edge_attr, edge_recon, config)
    loss.backward()
    optimizer.step()
    history['Total Loss'].append(loss.item()); history['Node MSE'].append(n_mse.item()); history['Edge MSE'].append(e_mse.item())
    if epoch % 50 == 0: print(f"Epoch {epoch:3d} | Loss: {loss.item():.4f}")

# 5. Save & Load (Verification of Persistence)
save_checkpoint(model, config, '../artifacts/model_v1', node_scaler=node_scaler, edge_scaler=edge_scaler)
loaded_model, loaded_config, l_node_scaler, l_edge_scaler = load_checkpoint(
    AMLGraphAutoencoder, '../artifacts/model_v1', train_data.num_node_features, train_data.num_edge_features, device=device
)

# 6. OOT Inference
oot_data = builder.build_graph(oot_df)
oot_data.x = torch.from_numpy(l_node_scaler.transform(oot_data.x.numpy())).float().to(device)
oot_data.edge_attr = torch.from_numpy(l_edge_scaler.transform(oot_data.edge_attr.numpy())).float().to(device)
oot_data.edge_index = oot_data.edge_index.to(device)

loaded_model.eval()
with torch.no_grad():
    z_oot, x_recon_oot, edge_recon_oot = loaded_model(oot_data.x, oot_data.edge_index, oot_data.edge_attr)
    node_errors = torch.mean((oot_data.x - x_recon_oot)**2, dim=1)
    edge_errors = torch.mean((oot_data.edge_attr - edge_recon_oot)**2, dim=1)

# 7. Deep Interpretation
interpreter = AnomalyInterpreter(loaded_model, loaded_config)

print("\n--- 1. Training Statistics ---")
interpreter.plot_loss_curves(history)

print("\n--- 2. SHAP Feature Importance (Node Profile) ---")
# Note: KernelSHAP is sampled for speed
interpreter.explain_node_with_shap(oot_data.x, oot_data.edge_index, oot_data.edge_attr)

print("\n--- 3. Local High-Risk Neighborhood Graph ---")
top_hub = int(torch.argmax(node_errors))
interpreter.local_perspective(
    top_hub, oot_data, node_errors, edge_errors, 
    num_hops=2, 
    inv_map=builder.inv_customer_map,
    min_edge_risk_quantile=0.80  # FILTER: Only show top 20% riskiest edges
)

print("\n--- 4. Exporting to anomalies.xlsx ---")
interpreter.save_anomalies_to_excel(oot_data, node_errors, edge_errors, oot_df, '../output/anomalies.xlsx', inv_map=builder.inv_customer_map)